## 1:2:2:1 Network Redux

This time I'm more confident than ever!!


Взять модель для линейной регрессии с двумя скрытыми слоями, в каждом по два нейрона. Для входных данных $x$ и целевого значения $y$ выписать выражения для:
1.  **Прямого прохода (Forward Pass)**
2.  **Обратного прохода (Backward Pass)**: градиентов функции потерь $L$ по всем весам.
3.  **Шага обновления весов**.

Сравнить три стратегии:
*   **Стандартное обучение (Standard Backpropagation)**
*   **Dropout** ($\sigma$)
*   **Gradient Dropout** ($\sigma^*$)

Проанализировать, как каждая стратегия влияет на минимизацию стандартных потерь $L$ и робастных потерь $L_{\text{robust}}$

In [85]:
from typing import Callable
import sympy as sp
from IPython.display import Latex, Markdown

class BaseBinding:
    def display(self) -> "Binding":
        display(Latex(f"${self.s}$ = {self.e._repr_latex_()}"))
        if self.doc:
            display(Markdown("⇡ " + self.doc))
        return self

def expand(expr: sp.Expr, bindings: list[BaseBinding]):
    return expr.subs([(b.s, b.e) for b in bindings])

def collapse(expr: sp.Expr, bindings: list[BaseBinding]):
    return expr.subs([(b.e, b.s) for b in bindings])


class Binding(BaseBinding):
    def __init__(self, name: str, e: sp.Expr, doc: str | None):
        self.s = sp.Symbol(name)
        self.e = e
        self.doc = doc

class MatBinding(BaseBinding):
    def __init__(
        self, n: int, m: int, name: str,
        e: Callable[[sp.MatrixExpr, sp.MatrixExpr], sp.MatrixExpr],
        doc: str | None
    ):
        self.s = sp.MatrixSymbol(name, m, n)
        self.e = sp.Matrix(m, n, e)
        self.doc = doc
        
    @property
    def shape(self) -> tuple[int, int]:
        return self.e.shape

class MatExprBinding(BaseBinding):
    def __init__(
        self, name: str,
        e: sp.MatrixExpr,
        doc: str | None
    ):
        m, n = e.shape
        self.s = sp.MatrixSymbol(name, m, n)
        self.e = e
        self.doc = doc

    @property
    def shape(self) -> tuple[int, int]:
        return self.e.shape

In [329]:
INPUT_NEURONS = 1
LAYER1_NEURONS = 2
LAYER2_NEURONS = 2
OUTPUT_NEURONS = 1

x = MatBinding(
    1, INPUT_NEURONS, "x",
    lambda i, j: sp.Symbol(f"x_{{{i + 1}}}"),
    doc="Neural network input.",
).display()

y = sp.Symbol("y")

W_1 = MatBinding(
    INPUT_NEURONS, LAYER1_NEURONS, "W_1",
    lambda i, j: sp.Symbol(f"w^{{(1)}}_{{{i + 1}{j + 1}}}"),
    doc="Weights of layer 1.",
).display()

b_1 = MatBinding(
    1, LAYER1_NEURONS, "b_1",
    lambda i, j: sp.Symbol(f'b^{{(1)}}_{{{i+1}}}'),
    doc="Biases of layer 1."
).display()

W_2 = MatBinding(
    LAYER2_NEURONS, LAYER1_NEURONS, "W_2",
    lambda i, j: sp.Symbol(f'w^{{(2)}}_{{{i+1}{j+1}}}'),
    doc="Weights of layer 2."
).display()

b_2 = MatBinding(
    1, LAYER2_NEURONS, "b_2",
    lambda i, j: sp.Symbol(f'b^{{(2)}}_{{{i+1}}}'),
    doc="Biases of layer 2."
).display()

W_3 = MatBinding(
    LAYER2_NEURONS, OUTPUT_NEURONS, "W_3",
    lambda i, j: sp.Symbol(f'w^{{(3)}}_{{{i+1}{j+1}}}'),
    doc="Weights of output layer."
).display()

b_3 = MatBinding(
    1, OUTPUT_NEURONS, "b_3",
    lambda i, j: sp.Symbol(f'b^{{(3)}}_{{{i+1}}}'),
    doc="Bias of the output layer."
).display()


<IPython.core.display.Latex object>

⇡ Neural network input.

<IPython.core.display.Latex object>

⇡ Weights of layer 1.

<IPython.core.display.Latex object>

...

⇡ Biases of layer 2.

<IPython.core.display.Latex object>

⇡ Weights of output layer.

<IPython.core.display.Latex object>

⇡ Bias of the output layer.


**Активации:** Для простоты анализа примем линейную функцию активации $\phi(z) = z$ для скрытых слоев. Это позволяет сосредоточиться на влиянии масок, не усложняя выкладки нелинейностями. Весь стек линейных слоев коллапсирует в один, но для анализа градиентов по *слоям* такая модель показательна.


In [327]:
def activate(x):
    return x

activate(x.s)

x

In [330]:
display(W_1.s * x.s)
display(W_1.e * x.e)
display(expand(W_1.s * x.s, [W_1, x]))
display(expand(W_1.s * x.s, [W_1, x]).doit())

W_1*x

Matrix([
[w^{(1)}_{11}*x_{1}],
[w^{(1)}_{21}*x_{1}]])

Matrix([
[w^{(1)}_{11}],
[w^{(1)}_{21}]])*Matrix([[x_{1}]])

Matrix([
[w^{(1)}_{11}*x_{1}],
[w^{(1)}_{21}*x_{1}]])


**Прямой проход (стандартный):**
1.  Выход 1-го скрытого слоя:
    $$z^{(1)} = W^{(1)}x + b^{(1)}$$
    $$a^{(1)} = \phi(z^{(1)}) = z^{(1)}$$
2.  Выход 2-го скрытого слоя:
    $$z^{(2)} = W^{(2)}a^{(1)} + b^{(2)}$$
    $$a^{(2)} = \phi(z^{(2)}) = z^{(2)}$$
3.  Выход сети (предсказание):
    $$\hat{y} = W^{(3)}a^{(2)} + b^{(3)}$$

In [331]:
z_1 = MatExprBinding(
    "z_1", W_1.s * x.s + b_1.s,
    doc="Layer 1 output."
).display()

a_1 = MatExprBinding(
    "a_1", activate(z_1.e),
    doc="Layer 1 activation."
).display()

z_2 = MatExprBinding(
    "z_2", W_2.s * a_1.s + b_2.s,
    doc="Layer 1 output."
).display()

a_2 = MatExprBinding(
    "a_1", activate(z_2.e),
    doc="Layer 2 activation."
).display()

# y_hat = MatExprBinding(
#     r'\hat{y}', W_3.s.T * a_2.s + b_3.s,
#     doc="Neural network output."
# ).display()

y_hat = Binding(
    r'\hat{y}', W_3.s * a_2.s + b_3.s,
    doc="Neural network output."
).display()

<IPython.core.display.Latex object>

⇡ Layer 1 output.

<IPython.core.display.Latex object>

⇡ Layer 1 activation.

<IPython.core.display.Latex object>

⇡ Layer 1 output.

<IPython.core.display.Latex object>

⇡ Layer 2 activation.

<IPython.core.display.Latex object>

⇡ Neural network output.

**Функция потерь:** Среднеквадратичная ошибка $L = \frac{1}{2}(\hat{y} - y)^2$.

In [332]:
L = Binding(
    "L", sp.Rational(1, 2) * (y_hat.s - y) ** sp.Integer(2),
    doc="Loss function."
).display()

<IPython.core.display.Latex object>

⇡ Loss function.


### Шаг 2: Анализ Standard Backpropagation (Базовая стратегия)

**1. Обратный проход:**
Начинаем с производной потерь по выходу сети:
$\delta^{(3)} = \frac{\partial L}{\partial \hat{y}} = (\hat{y} - y)$



In [333]:
delta_3 = Binding(
    r"\delta^{(3)}", sp.diff(L.e, y_hat.s), 
    doc="Output layer gradient."
).display()

<IPython.core.display.Latex object>

⇡ Output layer gradient.

Теперь распространяем ошибку назад по сети, используя цепное правило:
*   **Градиенты для выходного слоя:**
    $\nabla_{W^{(3)}} L = \delta^{(3)} (a^{(2)})^T$
    $\nabla_{b^{(3)}} L = \delta^{(3)}$

*   **Ошибка для 2-го скрытого слоя:**
    $\delta^{(2)} = (W^{(3)})^T \delta^{(3)} \odot \phi'(z^{(2)}) = (W^{(3)})^T \delta^{(3)}$ (т.к. $\phi'(z)=1$)
    *Здесь $\odot$ — поэлементное умножение.*

*   **Градиенты для 2-го скрытого слоя:**
    $\nabla_{W^{(2)}} L = \delta^{(2)} (a^{(1)})^T$
    $\nabla_{b^{(2)}} L = \delta^{(2)}$

*   **Ошибка для 1-го скрытого слоя:**
    $\delta^{(1)} = (W^{(2)})^T \delta^{(2)} \odot \phi'(z^{(1)}) = (W^{(2)})^T \delta^{(2)}$

*   **Градиенты для 1-го скрытого слоя:**
    $\nabla_{W^{(1)}} L = \delta^{(1)} x^T$
    $\nabla_{b^{(1)}} L = \delta^{(1)}$

In [335]:
L_epd = expand(L.e, [y_hat, a_1, W_3]).expand()
L_epd

y**2/2 + ((-y)*b_3 + Matrix([[-w^{(3)}_{11}*y, -w^{(3)}_{12}*y]])*W_1*x + Matrix([[-w^{(3)}_{11}*y, -w^{(3)}_{12}*y]])*b_1 + (1/2)*(Matrix([[w^{(3)}_{11}, w^{(3)}_{12}]])*W_1*x + Matrix([[w^{(3)}_{11}, w^{(3)}_{12}]])*b_1 + b_3)**2)

In [ ]:
# We can't diferentiate L_epd directly because of
# "Mix of Matrix and Scalar symbols" error so it's time for Manual Labor.
# Let's first remind ourselves what is W_3:
display(W_3.e)
display(W_3.e[0])
display(W_3.e[1])

Matrix([[w^{(3)}_{11}, w^{(3)}_{12}]])

w^{(3)}_{11}

w^{(3)}_{12}

'w^{(3)}_{11}'

In [388]:
"Let's split and analyze."

def disp(var_name, expr):
    display(Latex(fr"$$ {var_name} = {sp.latex(expr)}$$"))

def dispt(var_name, expr):
    display(Latex(fr"$$\text{{{var_name}}} = {sp.latex(expr)}$$"))

lp, rp = L_epd.args
dispt("lp + rp", L_epd)
dispt("lp", lp)
dispt("rp", rp)

for i, a in enumerate(rp.args):
    display(f"rp.args[{i}]: (W_3.e[0]: {a.has(W_3.e[0])}, W_3.e[1]: {a.has(W_3.e[0])})")
    display(a)

out = []
parts = [W_3.e[0], W_3.e[1]]
for part in parts:
    # We only need to consider rp.args[0], rp.args[2], and rp.args[3]
    display(Latex(
        r"${{\partial L} \over {" + sp.latex(part) + r"}} = " + sp.latex(rp.args[0] + rp.args[2] + rp.args[3]) + "$"
    ))

    display(Markdown(f"`rp.args[2]`  and `rp.args[3]`, differentiated by ${sp.latex(part)}$:"))
    rp2d0 = expand(rp.args[2].diff(part), [b_1]).doit()
    dispt("rp2d0", rp2d0)
    rp3d0 = expand(rp.args[3].diff(part), [W_1, x]).doit()
    dispt("rp3d0", rp3d0)

    display(rp.args[0])
    lp2, rp2 = rp.args[0].args
    dispt("lp2", lp2) # 1/2
    dispt("rp2", rp2) # the squared part
    lp3, rp3 = rp2.args
    dispt("lp3", lp3)
    dispt(r"rp3 \{square\}", rp3)
    display(Markdown("Let's rewrite `rp2` and save in `rp2r`:"))
    rp2r = sp.MatMul(rp2.args[0], rp2.args[0])
    dispt("rp2r", rp2r)
    rp0d0 = expand(rp2r.diff(W_3.e[0]), [W_1, x, b_1, b_3]).doit()
    display(Markdown(f"Now differentiate by ${sp.latex(part)}$"))
    dispt("rp0d0", rp0d0)

    expr = lp2 * (rp0d0 + rp2d0 + rp3d0).simplify()
    disp("{{\partial L} \over {" + sp.latex(part) + r"}}", expr)
    out.append(expr)


<>:47: SyntaxWarning: invalid escape sequence '\p'
<>:47: SyntaxWarning: invalid escape sequence '\p'
/var/folders/m9/y193wddj505gjbgjyvhjmzjm0000gq/T/ipykernel_76244/95754332.py:47: SyntaxWarning: invalid escape sequence '\p'
  disp("{{\partial L} \over {" + sp.latex(part) + r"}}", expr)


<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

'rp.args[0]: (W_3.e[0]: True, W_3.e[1]: True)'

...

Let's rewrite `rp2` and save in `rp2r`:

<IPython.core.display.Latex object>

Now differentiate by $w^{(3)}_{12}$

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

In [405]:
sp.srepr(sp.Matrix(out).T)

"MutableDenseMatrix([[Add(Mul(Integer(-1), Rational(1, 2), Symbol('b^{(1)}_{1}'), Symbol('y')), Mul(Integer(-1), Rational(1, 2), Symbol('w^{(1)}_{11}'), Symbol('x_{1}'), Symbol('y')), Mul(Add(Symbol('b^{(1)}_{1}'), Mul(Symbol('w^{(1)}_{11}'), Symbol('x_{1}'))), Add(Mul(Symbol('b^{(1)}_{1}'), Symbol('w^{(3)}_{11}')), Mul(Symbol('b^{(1)}_{2}'), Symbol('w^{(3)}_{12}')), Symbol('b^{(3)}_{1}'), Mul(Symbol('x_{1}'), Add(Mul(Symbol('w^{(1)}_{11}'), Symbol('w^{(3)}_{11}')), Mul(Symbol('w^{(1)}_{21}'), Symbol('w^{(3)}_{12}'))))))), Add(Mul(Integer(-1), Rational(1, 2), Symbol('b^{(1)}_{2}'), Symbol('y')), Mul(Integer(-1), Rational(1, 2), Symbol('w^{(1)}_{21}'), Symbol('x_{1}'), Symbol('y')), Mul(Add(Symbol('b^{(1)}_{1}'), Mul(Symbol('w^{(1)}_{11}'), Symbol('x_{1}'))), Add(Mul(Symbol('b^{(1)}_{1}'), Symbol('w^{(3)}_{11}')), Mul(Symbol('b^{(1)}_{2}'), Symbol('w^{(3)}_{12}')), Symbol('b^{(3)}_{1}'), Mul(Symbol('x_{1}'), Add(Mul(Symbol('w^{(1)}_{11}'), Symbol('w^{(3)}_{11}')), Mul(Symbol('w^{(1)}_{2

In [119]:
subs, res = sp.cse(y_hat)
display(subs)
display(res)

subs[1][1]

[(x0, b^{(1)}_{1} + w^{(1)}_{11}*x_{1} + w^{(1)}_{12}*x_{2}),
 (x1, b^{(1)}_{2} + w^{(1)}_{21}*x_{1} + w^{(1)}_{22}*x_{2})]

[Matrix([[b^{(3)}_{1} + w^{(3)}_{11}*(b^{(2)}_{1} + w^{(2)}_{11}*x0 + w^{(2)}_{12}*x1) + w^{(3)}_{12}*(b^{(2)}_{2} + w^{(2)}_{21}*x0 + w^{(2)}_{22}*x1)]])]

b^{(1)}_{2} + w^{(1)}_{21}*x_{1} + w^{(1)}_{22}*x_{2}

In [102]:
sp.cse(L)

([(x0, b^{(1)}_{1} + w^{(1)}_{11}*x_{1} + w^{(1)}_{12}*x_{2}),
  (x1, b^{(1)}_{2} + w^{(1)}_{21}*x_{1} + w^{(1)}_{22}*x_{2})],
 [Matrix([[(b^{(3)}_{1} + w^{(3)}_{11}*(b^{(2)}_{1} + w^{(2)}_{11}*x0 + w^{(2)}_{12}*x1) + w^{(3)}_{12}*(b^{(2)}_{2} + w^{(2)}_{21}*x0 + w^{(2)}_{22}*x1) - y)**2/2]])])

I have two expressions:

```
y_hat = Matrix([[b^{(3)}_{1} + w^{(3)}_{11}*(b^{(2)}_{1} + w^{(2)}_{11}*(b^{(1)}_{1} + w^{(1)}_{11}*x_{1} + w^{(1)}_{12}*x_{2}) + w^{(2)}_{12}*(b^{(1)}_{2} + w^{(1)}_{21}*x_{1} + w^{(1)}_{22}*x_{2})) + w^{(3)}_{12}*(b^{(2)}_{2} + w^{(2)}_{21}*(b^{(1)}_{1} + w^{(1)}_{11}*x_{1} + w^{(1)}_{12}*x_{2}) + w^{(2)}_{22}*(b^{(1)}_{2} + w^{(1)}_{21}*x_{1} + w^{(1)}_{22}*x_{2}))]])
L = Matrix([[(b^{(3)}_{1} + w^{(3)}_{11}*(b^{(2)}_{1} + w^{(2)}_{11}*(b^{(1)}_{1} + w^{(1)}_{11}*x_{1} + w^{(1)}_{12}*x_{2}) + w^{(2)}_{12}*(b^{(1)}_{2} + w^{(1)}_{21}*x_{1} + w^{(1)}_{22}*x_{2})) + w^{(3)}_{12}*(b^{(2)}_{2} + w^{(2)}_{21}*(b^{(1)}_{1} + w^{(1)}_{11}*x_{1} + w^{(1)}_{12}*x_{2}) + w^{(2)}_{22}*(b^{(1)}_{2} + w^{(1)}_{21}*x_{1} + w^{(1)}_{22}*x_{2})) - y)**2/2]])
```

I want to differentiate { \partial L \over \partial y_hat }. But it won't work directly, because L doesn't contain L explicitly.

I noticed my expressions can be rewritten as:

```
Q = b^{(3)}_{1} + w^{(3)}_{11}*(b^{(2)}_{1} + w^{(2)}_{11}*(b^{(1)}_{1} + w^{(1)}_{11}*x_{1} + w^{(1)}_{12}*x_{2}) + w^{(2)}_{12}*(b^{(1)}_{2} + w^{(1)}_{21}*x_{1} + w^{(1)}_{22}*x_{2}))
R = w^{(3)}_{12}*(b^{(2)}_{2} + w^{(2)}_{21}*(b^{(1)}_{1} + w^{(1)}_{11}*x_{1} + w^{(1)}_{12}*x_{2}) + w^{(2)}_{22}*(b^{(1)}_{2} + w^{(1)}_{21}*x_{1} + w^{(1)}_{22}*x_{2}))

L = Q + R
y_hat = (Q + R - y)**2/2
```

How do I differentiate??